# Flatmate RL Terminal-Reward GRPO

This notebook trains a stepwise policy with a terminal reward.
The model still predicts one next action at a time, but the reward is computed after replaying the full flow from that prefix to the end.

Keep the scenario list and max training step at the top of the notebook.

In [ ]:
from __future__ import annotations

SPACE_HTTP_URL = 'https://kushalexplores-flatmate-rl.hf.space'
SCENARIOS = [
    'task_visit_single',
    'task_visit_single_hidden_flex',
    'task_visit_multi',
    'task_visit_single_seller_followup',
]
MAX_TRAIN_STEP = 2
SEEDS_PER_SCENARIO = 5
START_SEED = 101
MAX_ROLLOUT_STEPS = 16
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'flatmate-grpo-terminal-reward-step2'
NUM_GENERATIONS = 8
GRPO_STEPS = 40
TERMINAL_REWARD_SCALE = 1.0
FORMAT_REWARD_WEIGHT = 0.25


In [ ]:
import asyncio
import inspect
import json
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import pandas as pd
import torch
import websockets
from datasets import Dataset
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from flatmate_rl import FlatmateRlAction
from flatmate_rl.server.heuristic_policy import expected_policy_action

SPACE_WS_URL = f"{'wss' if urlparse(SPACE_HTTP_URL).scheme == 'https' else 'ws'}://{urlparse(SPACE_HTTP_URL).netloc}/ws"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
RESULTS_DIR = Path(OUTPUT_DIR) / 'step_metrics'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(self.ws_url, open_timeout=self.timeout_s, ping_timeout=self.timeout_s)
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({'type': 'close'}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get('type') == 'error':
            raise RuntimeError(message.get('data', message))
        data = message['data']
        obs = data.get('observation', {})
        obs['reward'] = data.get('reward')
        obs['done'] = data.get('done', False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {'scenario_id': scenario_id}
        if seed is not None:
            data['seed'] = seed
        return await self._send({'type': 'reset', 'data': data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({'type': 'step', 'data': action})

def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    result: dict[str, Any] = {}
    def runner():
        try:
            result['value'] = asyncio.run(coro)
        except Exception as exc:
            result['error'] = exc
    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if 'error' in result:
        raise result['error']
    return result['value']

def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get('content', ''))
    return str(completion)

def parse_json_action(text: Any) -> dict[str, Any]:
    text = completion_text(text).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('completion does not contain a JSON object')
    action = json.loads(text[start : end + 1])
    if action.get('action_type') == 'assistant_message':
        msg = str(action.get('assistant_message', '')).strip()
        if not msg:
            raise ValueError('assistant_message action needs assistant_message')
        return {'action_type': 'assistant_message', 'assistant_message': msg}
    if action.get('action_type') == 'tool_call':
        tool_name = str(action.get('tool_name', '')).strip()
        if not tool_name:
            raise ValueError('tool_call action needs tool_name')
        args = action.get('tool_arguments', {})
        if not isinstance(args, dict):
            raise ValueError('tool_arguments must be an object')
        return {'action_type': 'tool_call', 'tool_name': tool_name, 'tool_arguments': args}
    raise ValueError("action_type must be assistant_message or tool_call")

def prompt_from_observation(obs: dict[str, Any], step_number: int) -> str:
    return json.dumps(
        {
            'step': step_number,
            'scenario_id': obs.get('scenario_id', ''),
            'phase': obs.get('phase', ''),
            'status': obs.get('status', ''),
            'remaining_required_fields': obs.get('remaining_required_fields', []),
            'available_tools': obs.get('available_tools', []),
            'feedback_summary': obs.get('feedback_summary', ''),
            'message': obs.get('message', ''),
            'last_tool_result': obs.get('last_tool_result', {}),
            'buyer_history': obs.get('buyer_conversation_history', [])[-4:],
            'seller_history': obs.get('seller_conversation_history', [])[-4:],
        },
        ensure_ascii=False,
        indent=2,
    )

print('imports ready')


In [ ]:
@dataclass
class CollectConfig:
    seeds_per_scenario: int = SEEDS_PER_SCENARIO
    start_seed: int = START_SEED
    max_train_step: int = MAX_TRAIN_STEP
    max_rollout_steps: int = MAX_ROLLOUT_STEPS

async def collect_rows(config: CollectConfig = CollectConfig()) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for scenario_id in SCENARIOS:
        for seed_offset in range(config.seeds_per_scenario):
            seed = config.start_seed + seed_offset
            prefix_actions: list[dict[str, Any]] = []
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                for step_idx in range(config.max_rollout_steps):
                    expected = expected_policy_action(scenario_id, obs)
                    if expected is None or obs.get('done'):
                        break
                    step_number = step_idx + 1
                    if step_number > config.max_train_step:
                        break
                    rows.append(
                        {
                            'prompt': prompt_from_observation(obs, step_number),
                            'scenario_id': scenario_id,
                            'seed': seed,
                            'episode_key': f'{scenario_id}:{seed}',
                            'step_idx': step_idx,
                            'step_number': step_number,
                            'prefix_actions_json': json.dumps(prefix_actions, ensure_ascii=False),
                            'reference_action_json': json.dumps(expected, ensure_ascii=False, sort_keys=True),
                        }
                    )
                    obs = await env.step(expected)
                    prefix_actions.append(expected)
                    if obs.get('done'):
                        break
    return rows

rows = run_async_blocking(collect_rows())
rows_df = pd.DataFrame(rows)
rows_df.to_csv(RESULTS_DIR / 'curriculum_rows.csv', index=False)
display(rows_df.groupby(['scenario_id', 'step_number']).size().unstack(fill_value=0))
train_rows = [row for row in rows if int(row['step_number']) <= MAX_TRAIN_STEP]
train_dataset = Dataset.from_list([
    {**row, 'prompt': [{'role': 'user', 'content': row['prompt']}]}
    for row in train_rows
])
train_dataset


In [ ]:
REWARD_DEBUG = {'parse_errors': [], 'endpoint_errors': [], 'valid': 0, 'invalid': 0}

def format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        try:
            parse_json_action(completion_text(completion))
            rewards.append(0.5)
            REWARD_DEBUG['valid'] += 1
        except Exception as exc:
            rewards.append(-1.0)
            REWARD_DEBUG['invalid'] += 1
            if len(REWARD_DEBUG['parse_errors']) < 20:
                REWARD_DEBUG['parse_errors'].append({'error': str(exc), 'completion': completion_text(completion)[:240]})
    return rewards

async def _score_one_completion(completion: Any, scenario_id: str, seed: int, prefix_actions_json: str) -> float:
    try:
        action = parse_json_action(completion_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -1.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get('done'):
                    return -0.25

            obs = await env.step(action)
            rollout_steps = 0
            while not obs.get('done') and rollout_steps < MAX_ROLLOUT_STEPS:
                next_payload = expected_policy_action(scenario_id, obs)
                if next_payload is None:
                    break
                obs = await env.step(next_payload)
                rollout_steps += 1

        reward = float(obs.get('total_reward') or obs.get('reward') or 0.0)
        reward *= TERMINAL_REWARD_SCALE
        if obs.get('done') and obs.get('status') != 'failed':
            reward += 0.5
        if obs.get('violations'):
            reward -= 0.5 * len(obs.get('violations', []))
        return float(max(-2.0, min(2.0, reward)))
    except Exception as exc:
        if len(REWARD_DEBUG['endpoint_errors']) < 20:
            REWARD_DEBUG['endpoint_errors'].append(repr(exc))
        return -1.0

async def _score_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    tasks = [
        _score_one_completion(c, s, int(sd), p)
        for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)
    ]
    return list(await asyncio.gather(*tasks))

def terminal_reward(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    return run_async_blocking(_score_batch(completions, scenario_id, seed, prefix_actions_json))

def make_grpo_config(**kwargs):
    valid = set(inspect.signature(GRPOConfig.__init__).parameters)
    filtered = {k: v for k, v in kwargs.items() if k in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print('Skipping unsupported GRPOConfig fields:', skipped)
    return GRPOConfig(**filtered)

print('reward functions ready')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if use_bf16 else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

grpo_args = make_grpo_config(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=200,
    max_steps=GRPO_STEPS,
    logging_steps=1,
    save_steps=max(10, GRPO_STEPS),
    save_total_limit=1,
    beta=0.1,
    temperature=1.0,
    top_p=0.95,
    report_to='none',
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, terminal_reward],
    args=grpo_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

# trainer.train()
# trainer.save_model(OUTPUT_DIR)
print('GRPO trainer configured for terminal reward training')
